In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder, PolynomialFeatures
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.ensemble import RandomForestRegressor

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['axes.grid'] = True

print("Done")


In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os

data_path = os.path.join(path, "Q1_data.csv")
print (data_path)

df = pd.read_csv(data_path)


In [ ]:
print("First 5 rows of the dataset:")
display(df.head())



In [ ]:
print("\nData info:")
print(df.info())



In [ ]:
print("\nStatistical summary for numeric columns:")
display(df.describe())

In [ ]:
plt.figure(figsize=(10, 5))
plt.hist(df['Delivery_Time'].dropna(), bins=50, edgecolor='black')
plt.title('Price Distribution')
plt.xlabel('Delivery_Time')
plt.ylabel('Frequency')
plt.show()

In [ ]:
df_model = df.copy()
df_model = df_model.drop(columns=["Order_ID"])

In [ ]:
print("Missing values per column before handling:")
print(df_model.isna().sum())

df_model = df_model.dropna(subset=["Delivery_Time"])
print("Shape after dropping rows with missing target:", df_model.shape)

In [ ]:
print("Shape before dropping duplicates:", df_model.shape)
df_model = df_model.drop_duplicates()
print("Shape after dropping duplicates:", df_model.shape)


In [ ]:

from sklearn.impute import SimpleImputer

numeric_features = ["Distance_km", "Preparation_Time_min", "Courier_Experience_yrs"]
categorical_features = ["Weather", "Traffic_Level", "Time_of_Day", "Vehicle_Type"]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)


In [ ]:

rf_model = RandomForestRegressor(random_state=42)

pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("scaler", StandardScaler(with_mean=False)),
    ("model", rf_model)
])


In [ ]:

y = df_model["Delivery_Time"]
print("Target description (Delivery_Time):")
print(y.describe())
print("Target imbalance check: not applicable for continuous regression target.")


In [ ]:

# Split the dataset into features (X) and target (y)
X = df_model.drop(columns=["Delivery_Time"])
print("Final feature columns:", X.columns.tolist())
print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:

kf = KFold(n_splits=5, shuffle=True, random_state=42)



In [ ]:

mae_scores = []
fold_index = 1

for train_idx, test_idx in kf.split(X):
    print(f"Starting fold {fold_index}")
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mae_scores.append(mae)
    print(f"Fold {fold_index} MAE: {mae:.4f}")
    fold_index += 1

print("MAE scores for each fold:", mae_scores)
print(f"Average MAE across folds: {np.mean(mae_scores):.4f}")


In [ ]:
# Plot feature importance from your trained model
rf_full = RandomForestRegressor(
    n_estimators=300,
    max_depth=None,
    random_state=42,
    n_jobs=-1
)

pipeline_full = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", rf_full)
])

print("Fitting RandomForest model on full data for feature importance...")
pipeline_full.fit(X, y)

preproc_fitted = pipeline_full.named_steps["preprocessor"]
feature_names = preproc_fitted.get_feature_names_out()
importances = pipeline_full.named_steps["model"].feature_importances_

sorted_idx = np.argsort(importances)[::-1]
top_n = 20 if len(feature_names) > 20 else len(feature_names)
top_idx = sorted_idx[:top_n]

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), importances[top_idx][::-1])
plt.yticks(range(top_n), [feature_names[i] for i in top_idx][::-1])
plt.xlabel("Feature importance")
plt.title("Top feature importances (RandomForestRegressor)")
plt.tight_layout()
plt.show()


In [ ]:

y_pred_full = pipeline_full.predict(X)

plt.figure(figsize=(8, 5))
plt.hist(y_pred_full, bins=30, edgecolor="black")
plt.xlabel("Predicted delivery time")
plt.ylabel("Frequency")
plt.title("Histogram of predicted delivery time")
plt.grid(True)
plt.show()




In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)
mae_scores_ensemble = []
fold_index = 1

for train_idx, val_idx in kf.split(X):
    print(f"Starting fold {fold_index}")
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Preprocess + scale inside each fold
    X_train_proc = preprocessor.fit_transform(X_train)
    X_val_proc = preprocessor.transform(X_val)

    scaler = StandardScaler(with_mean=False)
    X_train_scaled = scaler.fit_transform(X_train_proc)
    X_val_scaled = scaler.transform(X_val_proc)

    rf_model = RandomForestRegressor(
        n_estimators=300,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    )

    cb_model = CatBoostRegressor(
        depth=6,
        learning_rate=0.1,
        n_estimators=300,
        loss_function="RMSE",
        verbose=False,
        random_state=42
    )

    rf_model.fit(X_train_scaled, y_train)
    cb_model.fit(X_train_scaled, y_train)

    preds_rf = rf_model.predict(X_val_scaled)
    preds_cb = cb_model.predict(X_val_scaled)
    preds_avg = (preds_rf + preds_cb) / 2.0

    mae_fold = mean_absolute_error(y_val, preds_avg)
    mae_scores_ensemble.append(mae_fold)
    print(f"Fold {fold_index} MAE (ensemble): {mae_fold:.4f}")
    fold_index += 1

print("MAE scores for each fold (ensemble):", mae_scores_ensemble)
print(f"Average MAE across folds (ensemble): {np.mean(mae_scores_ensemble):.4f}")